<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration</b></h1>
</div>

## Theoretical Foundations

The implemented solution uses the pinhole-camera model, planar projective geometry, point normalization, normalized Direct Linear Transform (DLT), Singular Value Decomposition (SVD), Zhang's planar calibration method, closed-form intrinsic recovery, pose recovery and reprojection analysis. No lens-distortion model or nonlinear refinement is used.

## 1. Pinhole Camera Model

For a homogeneous world point $\mathbf{X}=[X,Y,Z,1]^T$ and homogeneous image point $\mathbf{x}=[u,v,1]^T$,

$$
s\mathbf{x}=K\begin{bmatrix}R&t\end{bmatrix}\mathbf{X}.
$$

The intrinsic matrix implemented in the notebook is

$$
K=
\begin{bmatrix}
\alpha & \gamma & u_0\\
0 & \beta & v_0\\
0 & 0 & 1
\end{bmatrix},
$$

where $\alpha$ and $\beta$ are focal scale factors in pixels, $\gamma$ is skew, and $(u_0,v_0)$ is the principal point.

## 2. Planar Calibration Geometry

The chessboard lies on $Z=0$. Its implementation coordinates are

$$
\mathbf{X}_p=[X,Y,1]^T,
$$

with the first internal corner at $(0,0)$, $X$ increasing along the 8-corner direction and $Y$ increasing along the 6-corner direction. Adjacent internal corners are separated by $0.03\,\mathrm{m}$.

For one view,

$$
s\mathbf{x}
=
K
\begin{bmatrix}
\mathbf{r}_1&\mathbf{r}_2&t
\end{bmatrix}
\mathbf{X}_p
=
H\mathbf{X}_p,
$$

so

$$
H=K\begin{bmatrix}\mathbf{r}_1&\mathbf{r}_2&t\end{bmatrix}.
$$

## 3. Point Normalization

The code normalizes the image points and planar points independently. For either 2D point set, let $(\bar x,\bar y)$ be the centroid and let $\bar d$ be the mean Euclidean distance from the centroid.

The scale is

$$
s_n=\frac{\sqrt{2}}{\bar d},
$$

and the similarity transform is

$$
T=
\begin{bmatrix}
s_n&0&-s_n\bar x\\
0&s_n&-s_n\bar y\\
0&0&1
\end{bmatrix}.
$$

This produces zero-centroid points with mean distance $\sqrt{2}$, matching `normalize_trans`.

## 4. Normalized DLT Homography

Each normalized planar-to-image correspondence $(X,Y)\leftrightarrow(u,v)$ contributes the two rows

$$
\begin{bmatrix}
X&Y&1&0&0&0&-uX&-uY&-u
\end{bmatrix},
$$

$$
\begin{bmatrix}
0&0&0&X&Y&1&-vX&-vY&-v
\end{bmatrix}
$$

to the matrix $Q$.

The homogeneous system is

$$
Q\mathbf{h}=0.
$$

With

$$
Q=U\Sigma V^T,
$$

the last right singular vector gives $\mathbf{h}$, which the implementation reshapes into the normalized homography $H_n$.

The exact denormalization used in code is

$$
H=T_{\mathrm{image}}^{-1}H_nT_{\mathrm{plane}},
$$

followed by

$$
H\leftarrow\frac{H}{H_{33}}.
$$

## 5. Zhang Intrinsic Constraints

Write the homography columns as

$$
H=[\mathbf{h}_1\;\mathbf{h}_2\;\mathbf{h}_3],
$$

and define

$$
B=K^{-T}K^{-1}.
$$

The first two rotation columns are orthogonal and have equal norm, giving

$$
\mathbf{h}_1^TB\mathbf{h}_2=0,
$$

$$
\mathbf{h}_1^TB\mathbf{h}_1-\mathbf{h}_2^TB\mathbf{h}_2=0.
$$

For homography columns $\mathbf{h}_i$ and $\mathbf{h}_j$, the implementation constructs

$$
v_{ij}=
\begin{bmatrix}
h_{i1}h_{j1}\\
h_{i1}h_{j2}+h_{i2}h_{j1}\\
h_{i2}h_{j2}\\
h_{i3}h_{j1}+h_{i1}h_{j3}\\
h_{i3}h_{j2}+h_{i2}h_{j3}\\
h_{i3}h_{j3}
\end{bmatrix}.
$$

Each valid view contributes

$$
v_{12},
\qquad
v_{11}-v_{22}.
$$

All constraints are stacked into

$$
Vb=0,
$$

with

$$
b=[b_{11},b_{12},b_{22},b_{13},b_{23},b_{33}]^T.
$$

The implementation solves this system by SVD and selects the last right singular vector.

## 6. Recovering the Intrinsic Matrix

Define

$$
d=b_{11}b_{22}-b_{12}^2,
$$

$$
v_0=\frac{b_{12}b_{13}-b_{11}b_{23}}{d},
$$

and

$$
\lambda=
b_{33}
-
\frac{b_{13}^2+v_0(b_{12}b_{13}-b_{11}b_{23})}{b_{11}}.
$$

Then

$$
\alpha=\sqrt{\frac{\lambda}{b_{11}}},
\qquad
\beta=\sqrt{\frac{\lambda b_{11}}{d}},
$$

$$
\gamma=-\frac{b_{12}\alpha^2\beta}{\lambda},
$$

$$
u_0=
\frac{\gamma v_0}{\beta}
-
\frac{b_{13}\alpha^2}{\lambda}.
$$

Because $b$ is homogeneous and therefore defined up to sign, the implementation flips $b\leftarrow-b$ when necessary before the square roots so that the required terms are positive.

These values form

$$
K=
\begin{bmatrix}
\alpha&\gamma&u_0\\
0&\beta&v_0\\
0&0&1
\end{bmatrix}.
$$

## 7. Camera Pose Recovery

For

$$
H=[\mathbf{h}_1\;\mathbf{h}_2\;\mathbf{h}_3],
$$

the implementation uses

$$
\lambda_p=
\frac{1}{\lVert K^{-1}\mathbf{h}_1\rVert},
$$

$$
\mathbf{r}_1=\lambda_pK^{-1}\mathbf{h}_1,
\qquad
\mathbf{r}_2=\lambda_pK^{-1}\mathbf{h}_2,
$$

$$
\mathbf{r}_3=\mathbf{r}_1\times\mathbf{r}_2,
\qquad
t=\lambda_pK^{-1}\mathbf{h}_3.
$$

The initial rotation estimate is

$$
R_{\mathrm{approx}}
=
[\mathbf{r}_1\;\mathbf{r}_2\;\mathbf{r}_3].
$$

If

$$
R_{\mathrm{approx}}=U\Sigma V^T,
$$

the nearest orthonormal rotation used by the implementation is

$$
R=UV^T.
$$

If $\det(R)<0$, the last column of $U$ is sign-flipped and $R$ is recomputed so that $\det(R)=+1$.

## 8. Reprojection

Each planar calibration point is embedded in 3D as

$$
\mathbf{X}_{3D}=[X,Y,0]^T.
$$

The implementation computes the camera-frame point

$$
\mathbf{X}_c=R\mathbf{X}_{3D}+t,
$$

then the homogeneous image point

$$
\tilde{\mathbf{x}}=K\mathbf{X}_c.
$$

Cartesian pixel coordinates are obtained by dividing the first two homogeneous components by the third.

## 9. Reprojection Error

For measured pixel point $\mathbf{x}_i$ and predicted pixel point $\hat{\mathbf{x}}_i$,

$$
e_i=\lVert\hat{\mathbf{x}}_i-\mathbf{x}_i\rVert_2.
$$

For one view containing $N$ corners,

$$
\bar e=\frac{1}{N}\sum_{i=1}^{N}e_i,
$$

$$
\mathrm{RMSE}=\sqrt{\frac{1}{N}\sum_{i=1}^{N}e_i^2}.
$$

The same definitions are applied globally after concatenating the residual magnitudes from all valid views.

## 10. Camera Centre Used in the Pose Figure

The pose visualization plots each camera centre in world coordinates. From

$$
\mathbf{X}_c=R\mathbf{X}_w+t,
$$

the camera centre is

$$
C=-R^Tt.
$$

This is exactly the quantity plotted in `estimated_camera_poses.png`.

## Theory-to-Code Correspondence

| Mathematical object | Implementation |
| --- | --- |
| $T_{\mathrm{image}}$, $T_{\mathrm{plane}}$ | `normalize_trans` |
| $Q$ | `Image.find_homography` |
| $H_n$ | `H_normalized` |
| $H$ | `Image.H` |
| $v_{ij}$ | `Image._v_ij` |
| $V$ | stacked `image.construct_v()` |
| $b$ | last row of `Vt` from SVD of $V$ |
| $\alpha,\beta,\gamma,u_0,v_0$ | intrinsic-recovery cell |
| $K$ | `K` |
| $R,t$ | `Image.find_extrinsic(K)` |
| $\hat{\mathbf{x}}_i$ | `projected_pixels` |
| $e_i$ | `errors` |
| $\bar e$ | `mean_error` / `overall_mean_error` |
| RMSE | `rmse` / `overall_rmse` |
| $C=-R^Tt$ | `camera_center` |

## Scope and Limitations

- The target is planar.
- Chessboard geometry is assumed known.
- Correspondences come from successful OpenCV chessboard detection.
- At least three geometrically distinct valid views are required by the implementation.
- The implemented camera model contains no radial or tangential lens-distortion terms.
- No nonlinear refinement is performed after the closed-form calibration.
- Reprojection residuals therefore evaluate the exact pinhole-model solution implemented in the companion notebook.